# 02 - Bronze to Silver

Reads source-shaped Bronze tables and writes schema-enforced, deduplicated, conformed Silver Delta tables. Public-reference dimensions retain provenance; synthetic operations are standardized to UTC with derived local timestamps using IANA time zones.

Notebook 00 injects the default Lakehouse. Reruns overwrite owned Silver tables without duplicating records.

In [ ]:
from pyspark.sql import functions as F


def save_silver(frame, name):
    frame = frame.withColumn('data_classification', F.lit('DerivedAnalytical'))
    if 'data_quality_status' not in frame.columns:
        frame = frame.withColumn('data_quality_status', F.lit('Valid'))
    if 'rejection_reason' not in frame.columns:
        frame = frame.withColumn('rejection_reason', F.lit(None).cast('string'))
    frame.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(name)
    print('silver', name, frame.count())

In [ ]:
# Conformed public-reference and synthetic master dimensions/bridges.
save_silver(spark.table('bronze_country').dropDuplicates(['country_id']), 'dim_country')
save_silver(spark.table('bronze_airport').dropDuplicates(['airport_id']), 'dim_airport')
save_silver(spark.table('bronze_airline').dropDuplicates(['airline_id']), 'dim_airline')
save_silver(spark.table('bronze_aircraft').dropDuplicates(['aircraft_type_id']), 'dim_aircraft')
save_silver(spark.table('bronze_gate').dropDuplicates(['gate_id']), 'dim_gate')
save_silver(spark.table('bronze_service_team').dropDuplicates(['team_id']), 'dim_service_team')
save_silver(spark.table('bronze_runway_reference').dropDuplicates(['runway_reference_id']), 'dim_runway_reference')
save_silver(spark.table('bronze_airport_group_assignment').dropDuplicates(['airport_id']), 'bridge_airport_group_assignment')
save_silver(spark.table('bronze_airline_aircraft_eligibility').dropDuplicates(['eligibility_id']), 'bridge_airline_aircraft_eligibility')
save_silver(spark.table('bronze_airline_airport_service').dropDuplicates(['service_id']), 'bridge_airline_airport_service')

In [ ]:
# Flight fact: scheduled/estimated/actual UTC and local time, turnaround, and delay measures.
flight_source = spark.table('bronze_flight_turnaround').dropDuplicates(['flight_event_id'])
airport_time_zones = spark.table('dim_airport').select('airport_id','iana_time_zone')
flt = (flight_source.join(airport_time_zones, 'airport_id', 'inner')
       .withColumn('scheduled_arrival_utc', F.col('scheduled_arrival').cast('timestamp'))
       .withColumn('estimated_arrival_utc', F.col('estimated_arrival').cast('timestamp'))
       .withColumn('actual_arrival_utc', F.col('actual_arrival').cast('timestamp'))
       .withColumn('scheduled_departure_utc', F.col('scheduled_departure').cast('timestamp'))
       .withColumn('estimated_departure_utc', F.col('estimated_departure').cast('timestamp'))
       .withColumn('actual_departure_utc', F.col('actual_departure').cast('timestamp'))
       .withColumn('scheduled_arrival_local', F.expr('from_utc_timestamp(scheduled_arrival_utc, iana_time_zone)'))
       .withColumn('estimated_arrival_local', F.expr('from_utc_timestamp(estimated_arrival_utc, iana_time_zone)'))
       .withColumn('actual_arrival_local', F.expr('from_utc_timestamp(actual_arrival_utc, iana_time_zone)'))
       .withColumn('scheduled_departure_local', F.expr('from_utc_timestamp(scheduled_departure_utc, iana_time_zone)'))
       .withColumn('estimated_departure_local', F.expr('from_utc_timestamp(estimated_departure_utc, iana_time_zone)'))
       .withColumn('actual_departure_local', F.expr('from_utc_timestamp(actual_departure_utc, iana_time_zone)'))
       .withColumn('turnaround_minutes', (F.col('turnaround_end').cast('long') - F.col('turnaround_start').cast('long')) / 60)
       .withColumn('arrival_delay_minutes', (F.col('actual_arrival_utc').cast('long') - F.col('scheduled_arrival_utc').cast('long')) / 60)
       .withColumn('departure_delay_minutes', (F.col('actual_departure_utc').cast('long') - F.col('scheduled_departure_utc').cast('long')) / 60)
       .withColumn('arrival_estimate_error_minutes', (F.col('actual_arrival_utc').cast('long') - F.col('estimated_arrival_utc').cast('long')) / 60)
       .withColumn('departure_estimate_error_minutes', (F.col('actual_departure_utc').cast('long') - F.col('estimated_departure_utc').cast('long')) / 60)
       .withColumn('on_time_arrival_flag', (F.col('arrival_delay_minutes') <= 15).cast('int'))
       .withColumn('on_time_flag', (F.col('departure_delay_minutes') <= 15).cast('int'))
       .withColumn('date_key', F.date_format('scheduled_departure_utc', 'yyyyMMdd').cast('int'))
       .withColumn('arrival_date_key', F.date_format('scheduled_arrival_utc', 'yyyyMMdd').cast('int'))
       .withColumn('actual_departure_date_key', F.date_format('actual_departure_utc', 'yyyyMMdd').cast('int'))
       .withColumn('event_hour', F.hour('scheduled_departure_utc')))
save_silver(flt, 'fact_flight_turnaround_events')

In [ ]:
# --- fact_passenger_queue_metrics ---
q = (spark.table('bronze_passenger_queue')
     .dropDuplicates(['queue_metric_id'])
     .withColumn('date_key', F.date_format('event_time', 'yyyyMMdd').cast('int'))
     .withColumn('event_hour', F.hour('event_time')))
save_silver(q, 'fact_passenger_queue_metrics')

In [ ]:
# --- fact_energy_metering ---
en = (spark.table('bronze_energy')
      .dropDuplicates(['meter_reading_id'])
      .withColumn('date_key', F.date_format('event_time', 'yyyyMMdd').cast('int'))
      .withColumn('event_hour', F.hour('event_time')))
save_silver(en, 'fact_energy_metering')

In [ ]:
# --- fact_maintenance_events (typed anomaly flag) ---
mt = (spark.table('bronze_maintenance')
      .dropDuplicates(['maintenance_id'])
      .withColumn('anomaly_flag', F.col('anomaly_flag').cast('boolean'))
      .withColumn('date_key', F.date_format('event_time', 'yyyyMMdd').cast('int'))
      .withColumn('event_hour', F.hour('event_time')))
save_silver(mt, 'fact_maintenance_events')

In [ ]:
# --- fact_weather ---
wx = (spark.table('bronze_weather')
      .dropDuplicates(['weather_id'])
      .withColumn('date_key', F.date_format('event_time', 'yyyyMMdd').cast('int'))
      .withColumn('event_hour', F.hour('event_time')))
save_silver(wx, 'fact_weather')

In [ ]:
# --- fact_operational_incidents ---
inc = (spark.table('bronze_operational_incidents')
       .dropDuplicates(['incident_id'])
       .withColumn('date_key', F.date_format('event_time', 'yyyyMMdd').cast('int'))
       .withColumn('event_hour', F.hour('event_time')))
save_silver(inc, 'fact_operational_incidents')
print('Silver layer complete.')